In [ ]:
import pandas as pd
import os as os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ==== Classification metrics ====
import pandas as pd

import pandas as pd

import pandas as pd

# Common metric order
metrics = ["Weighted-F1", "Macro-F1", "Micro-F1 ", "Sample-F1", "Jaccard Index", "Hamming Loss"]

# df1 full data, rounded to 4 decimals, unified column names
df1 = pd.DataFrame({
    "LGBM_MOC train": [0.6299, 0.3487, 0.6507, 0.6609, 0.6046, 0.1714],
    "LGBM_MOC test":  [0.6268, 0.3525, 0.6474, 0.6569, 0.6015, 0.1726],
    "LGBM_CC train":  [0.6139, 0.3154, 0.6504, 0.6551, 0.6203, 0.1587],
    "LGBM_CC test":   [0.6123, 0.3192, 0.6489, 0.6526, 0.6207, 0.1580],
    "MLRF train":   [0.6143, 0.3015, 0.6600, 0.6648, 0.6168, 0.1596],
    "MLRF test":    [0.5990, 0.2946, 0.6445, 0.6479, 0.5990, 0.1672],
}, index=metrics).round(4)

# df2 full data, rounded to 4 decimals, unified column names
df2 = pd.DataFrame({
    "LGBM_MOC test":  [0.6509, 0.4152, 0.6623, 0.6725, 0.6218, 0.1633],
    "LGBM_MOC train": [0.7831, 0.7112, 0.7840, 0.8010, 0.7504, 0.1049],
    "RF_MOC test":    [0.6295, 0.3242, 0.6692, 0.6695, 0.6352, 0.1494],
    "RF_MOC train":   [0.7858, 0.4293, 0.8241, 0.8236, 0.7959, 0.0775],
    "LGBM_CC test":   [0.6423, 0.4011, 0.6571, 0.6652, 0.6342, 0.1563],
    "LGBM_CC train":  [0.7391, 0.6570, 0.7437, 0.7518, 0.7233, 0.1171],
    "RF_CC test":     [0.6479, 0.4122, 0.6556, 0.6662, 0.6323, 0.1591],
    "RF_CC train":    [0.8566, 0.7004, 0.8596, 0.8775, 0.8514, 0.0640],
    "MLRF test":    [0.6361, 0.3611, 0.6670, 0.6738, 0.6339, 0.1543],
    "MLRF train":   [0.8394, 0.6218, 0.8536, 0.8660, 0.8355, 0.0663],
}, index=metrics).round(4)


df1.columns = [f"{col} (D1)" for col in df1.columns]
df2.columns = [f"{col} (D2)" for col in df2.columns]

df_all = pd.concat([df1, df2], axis=1)
train_cols = [col for col in df_all.columns if "train" in col.lower()]
test_cols  = [col for col in df_all.columns if "test" in col.lower()]
df_train = df_all[train_cols]
df_test = df_all[test_cols]

# ==== Normalization for colors ====
def normalize_classification(df):
    df_norm = df.copy()
    for row in df_norm.index:
        vals = df.loc[row].values.reshape(-1, 1)
        scaler = MinMaxScaler()
        
        # Lower-is-better metrics (Hamming Loss) are inverted after normalization
        lower_better = ["Hamming Loss"]

        if row in lower_better:
            df_norm.loc[row] = 1 - scaler.fit_transform(vals).flatten()
        else:
            df_norm.loc[row] = scaler.fit_transform(vals).flatten()
    return df_norm
df_train_norm = normalize_classification(df_train)
df_test_norm  = normalize_classification(df_test)

# ==== Visualization ====
def plot_horizontal_heatmap(df_train_raw, df_test_raw, df_train_norm, df_test_norm, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(df_train_raw.columns) * 0.45)), sharey=True)

    for ax, df_raw, df_norm, title in zip(
        axes, [df_train_raw, df_test_raw], [df_train_norm, df_test_norm], ["(a) Train Performance", "(b) Test Performance"]
    ):
        df_plot = df_raw.T
        df_annot = df_plot.applymap(lambda x: f"{x:.4f}")
        df_color = df_norm.T

         # Two-step approach: strip train/test, keep model name and (D1)/(D2)
        y_labels = df_plot.index.str.replace(' train ', ' ', regex=False)
        y_labels = y_labels.str.replace(' test ', ' ', regex=False)

        sns.heatmap(
            df_color,
            annot=df_annot,
            fmt="",
            cmap="RdBu",
            vmin=0,
            vmax=1,
            linewidths=0.5,
            ax=ax,
            cbar=False,
            annot_kws={"size": 10},   # heatmap annotation font size
            xticklabels=True,
            yticklabels=y_labels 
        )
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Metric", fontsize=14)     # x-axis label font size
        ax.set_ylabel("Model", fontsize=14)      # y-axis label font size
        ax.tick_params(axis='x', labelsize=12, labelrotation=45)   # x-axis tick font size
        ax.tick_params(axis='y', labelsize=12)   # y-axis tick font size


    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Bad", "Good"])
    cbar.ax.tick_params(labelsize=12)           # colorbar tick font size
    cbar.set_label("Performance Scale", fontsize=13)  # colorbar label font size

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()
# ==== Call ====
plot_horizontal_heatmap(df_train, df_test, df_train_norm, df_test_norm,
                        save_path="results/figures/classification_heatmap.png")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ==== Classification metrics ====
import pandas as pd


# Common metric order
metrics = ["Weighted-F1", "Macro-F1", "Micro-F1 ", "Sample-F1", "Jaccard Index", "Hamming Loss"]

# df1 full data, rounded to 4 decimals, unified column names
df1 = pd.DataFrame({
    "LGBM_MOC train": [0.6299, 0.3487, 0.6507, 0.6609, 0.6046, 0.1714],
    "LGBM_MOC test":  [0.6268, 0.3525, 0.6474, 0.6569, 0.6015, 0.1726],
    "LGBM_CC train":  [0.6139, 0.3154, 0.6504, 0.6551, 0.6203, 0.1587],
    "LGBM_CC test":   [0.6123, 0.3192, 0.6489, 0.6526, 0.6207, 0.1580],
    "MLRF train":   [0.6143, 0.3015, 0.6600, 0.6648, 0.6168, 0.1596],
    "MLRF test":    [0.5990, 0.2946, 0.6445, 0.6479, 0.5990, 0.1672],
}, index=metrics).round(4)

# df2 full data, rounded to 4 decimals, unified column names
df2 = pd.DataFrame({
    # ---- LGBM_MOC ----
    "LGBM_MOC test":  [0.6439, 0.3819, 0.6675, 0.6747, 0.6220, 0.1600],
    "LGBM_MOC train": [0.8109, 0.7413, 0.8092, 0.8278, 0.7771, 0.0926],
    # ---- RF_MOC ----
    "RF_MOC test":    [0.6278, 0.3435, 0.6590, 0.6653, 0.6148, 0.1630],
    "RF_MOC train":   [0.8076, 0.5304, 0.8227, 0.8395, 0.7943, 0.0839],
    # ---- LGBM_CC ----
    "LGBM_CC test":   [0.6366, 0.3937, 0.6540, 0.6648, 0.6223, 0.1641],
    "LGBM_CC train":  [0.7399, 0.6503, 0.7442, 0.7586, 0.7178, 0.1214],
    # ---- RF_CC ----
    "RF_CC test":     [0.6428, 0.4217, 0.6423, 0.6570, 0.6160, 0.1704],
    "RF_CC train":    [0.8827, 0.8260, 0.8826, 0.9020, 0.8776, 0.0533],
    # ---- MLRF (placed last) ----
    "MLRF test":      [0.6294, 0.3467, 0.6529, 0.6638, 0.6144, 0.1663],
    "MLRF train":     [0.8251, 0.5263, 0.8486, 0.8620, 0.8279, 0.0694],
}, index=metrics).round(4)



df1.columns = [f"{col} (D1)" for col in df1.columns]
df2.columns = [f"{col} (D2)" for col in df2.columns]

df_all = pd.concat([df1, df2], axis=1)
train_cols = [col for col in df_all.columns if "train" in col.lower()]
test_cols  = [col for col in df_all.columns if "test" in col.lower()]
df_train = df_all[train_cols]
df_test = df_all[test_cols]

# ==== Normalization for colors ====
def normalize_classification(df):
    df_norm = df.copy()
    for row in df_norm.index:
        vals = df.loc[row].values.reshape(-1, 1)
        scaler = MinMaxScaler()
        
        # Lower-is-better metrics (Hamming Loss) are inverted after normalization
        lower_better = ["Hamming Loss"]

        if row in lower_better:
            df_norm.loc[row] = 1 - scaler.fit_transform(vals).flatten()
        else:
            df_norm.loc[row] = scaler.fit_transform(vals).flatten()
    return df_norm
df_train_norm = normalize_classification(df_train)
df_test_norm  = normalize_classification(df_test)

# ==== Visualization ====
def plot_horizontal_heatmap(df_train_raw, df_test_raw, df_train_norm, df_test_norm, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(df_train_raw.columns) * 0.45)), sharey=True)

    for ax, df_raw, df_norm, title in zip(
        axes, [df_train_raw, df_test_raw], [df_train_norm, df_test_norm], ["(a) Train Performance", "(b) Test Performance"]
    ):
        df_plot = df_raw.T
        df_annot = df_plot.applymap(lambda x: f"{x:.4f}")
        df_color = df_norm.T

         # Two-step approach: strip train/test, keep model name and (D1)/(D2)
        y_labels = df_plot.index.str.replace(' train ', ' ', regex=False)
        y_labels = y_labels.str.replace(' test ', ' ', regex=False)

        sns.heatmap(
            df_color,
            annot=df_annot,
            fmt="",
            cmap="RdBu",
            vmin=0,
            vmax=1,
            linewidths=0.5,
            ax=ax,
            cbar=False,
            annot_kws={"size": 10},   # heatmap annotation font size
            xticklabels=True,
            yticklabels=y_labels 
        )
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Metric", fontsize=14)     # x-axis label font size
        ax.set_ylabel("Model", fontsize=14)      # y-axis label font size
        ax.tick_params(axis='x', labelsize=12, labelrotation=45)   # x-axis tick font size
        ax.tick_params(axis='y', labelsize=12)   # y-axis tick font size


    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Bad", "Good"])
    cbar.ax.tick_params(labelsize=12)           # colorbar tick font size
    cbar.set_label("Performance Scale", fontsize=13)  # colorbar label font size

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()
# ==== Call ====
plot_horizontal_heatmap(df_train, df_test, df_train_norm, df_test_norm,
                        save_path="results/figures/classification_heatmap.png")